# Notebook 01: Exploratory Data Analysis (EDA) — MVTec AD Bottle

**Objective:** Inspect, profile, and visually validate the MVTec Anomaly Detection (AD) `bottle` category.

### Core Learning Objectives:
1. Understand the directory layout of an industrial anomaly benchmark.
2. Differentiate between training split (normal-only) and testing split (normal + anomalous).
3. Inspect pixel-level ground truth masks and measure defect area distributions.
4. Formulate the PyTorch Dataset structure and verify data loading with zero data leakage.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import MVTEC_BOTTLE_DIR
from src.data.dataset import MVTecDataset
from src.data.eda import scan_dataset_structure, analyze_image_properties, analyze_mask_properties

print(f"MVTec Bottle Path: {MVTEC_BOTTLE_DIR}")
print(f"Dataset exists on disk: {MVTEC_BOTTLE_DIR.exists()}")

## 1. Dataset Directory Hierarchy and Sample Distribution

Let's scan the dataset folders and create a unified catalog of all samples.

In [ ]:
df = scan_dataset_structure(MVTEC_BOTTLE_DIR)
print(f"Total images indexed: {len(df)}")

summary = df.groupby(["split", "defect_type", "is_anomaly"]).size().reset_index(name="count")
display(summary)

In [ ]:
# Visualise sample counts per split
counts_table = df.groupby(["split", "defect_type"]).size().unstack(fill_value=0)
counts_table.plot(kind="bar", stacked=True, figsize=(8, 4.5), edgecolor="black", colormap="viridis")
plt.title("MVTec AD Bottle: Sample Distribution Across Splits")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

## 2. Image Physical Properties (Resolution, Color Space, Statistics)

We inspect image resolution, color channels, and RGB mean/std intensity.

In [ ]:
props = analyze_image_properties(df)
print(f"Unique Image Resolutions (W x H): {props['unique_resolutions']}")
print(f"Color Channels: {props['unique_channels']}")
print(f"Dataset RGB Mean: {[round(x, 4) for x in props['dataset_rgb_mean']]}")
print(f"Dataset RGB Std:  {[round(x, 4) for x in props['dataset_rgb_std']]}")

## 3. Defect Mask Analysis (Pixel Defect Area % and Connected Components)

For anomalous test images, we measure:
- Exact pixel defect count
- Proportion of the image occupied by the defect (`defect_pct`)
- Number of separate defect clusters (connected components)

In [ ]:
mask_df = analyze_mask_properties(df)
stats_by_type = mask_df.groupby("defect_type").agg(
    sample_count=("defect_pct", "count"),
    min_defect_pct=("defect_pct", "min"),
    mean_defect_pct=("defect_pct", "mean"),
    max_defect_pct=("defect_pct", "max"),
    mean_components=("num_components", "mean"),
    max_components=("num_components", "max")
).reset_index()

display(stats_by_type)

In [ ]:
# Defect area percentage histogram
plt.figure(figsize=(8, 4))
for dtype, grp in mask_df.groupby("defect_type"):
    plt.hist(grp["defect_pct"], bins=12, alpha=0.6, label=dtype, edgecolor="black")
plt.title("Defect Pixel Area Distribution (% of Total Image)")
plt.xlabel("Defect Area (%)")
plt.ylabel("Count")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

## 4. Visual Inspection Gallery (Original vs Mask vs Overlay)

We overlay ground-truth defect masks onto the product images in translucent red to inspect defect morphology.

In [ ]:
defect_types = mask_df["defect_type"].unique()
fig, axes = plt.subplots(len(defect_types), 3, figsize=(11, 3.8 * len(defect_types)))

for i, dtype in enumerate(defect_types):
    sample = df[(df["defect_type"] == dtype) & (df["mask_path"].notnull())].iloc[0]
    img_bgr = cv2.imread(sample["image_path"])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(sample["mask_path"], cv2.IMREAD_GRAYSCALE)
    
    # Red highlight overlay
    overlay = img_rgb.copy()
    overlay[mask > 0] = [255, 30, 30]
    blended = cv2.addWeighted(img_rgb, 0.65, overlay, 0.35, 0)
    
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f"{dtype} - Original Image")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title(f"{dtype} - Ground Truth Mask")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(blended)
    axes[i, 2].set_title(f"{dtype} - Red Defect Overlay")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## 5. PyTorch Dataset & DataLoader Smoke Test

Verify that `MVTecDataset` outputs correct shapes, data types, and batches through `DataLoader`.

In [ ]:
train_ds = MVTecDataset(category_dir=MVTEC_BOTTLE_DIR, split="train", img_size=256)
test_ds = MVTecDataset(category_dir=MVTEC_BOTTLE_DIR, split="test", img_size=256)

print(f"Train dataset length: {len(train_ds)}")
print(f"Test dataset length:  {len(test_ds)}")

loader = DataLoader(test_ds, batch_size=4, shuffle=True)
batch = next(iter(loader))

print(f"Batch Image Tensor Shape: {batch['image'].shape} (dtype: {batch['image'].dtype})")
print(f"Batch Mask Tensor Shape:  {batch['mask'].shape} (dtype: {batch['mask'].dtype})")
print(f"Batch Labels:              {batch['label'].tolist()}")
print(f"Batch Defect Types:        {batch['defect_type']}")